In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/kylin0338/tuningandtesting/label_test.csv.xlsx
/kaggle/input/datasets/kylin0338/tuningandtesting/label_tune_labeled.xlsx
/kaggle/input/datasets/kylin0338/notupdatedlabeltune/label_tune.csv
/kaggle/input/datasets/kylin0338/aspectsentimentoutput/aspect_sentiment.csv
/kaggle/input/datasets/kylin0338/pairsversion2/pairs_with_probs_v2.csv
/kaggle/input/datasets/kylin0338/updated-validation/validation.csv.xlsx
/kaggle/input/datasets/kylin0338/labeltestv2/label_test_v2_LABELED.csv
/kaggle/input/datasets/kylin0338/validationfixed/validation_fixed.xlsx


In [2]:
import pandas as pd, numpy as np

v = pd.read_excel("/kaggle/input/datasets/kylin0338/updated-validation/validation.csv.xlsx")

# where the prediction was right, the true label equals the prediction
v["true_sentiment"] = np.where(v["sentiment_correct"] == 1, v["aspect_sentiment"], "")
print((v["true_sentiment"] == "").sum(), "rows need a manual label")  # should be ~55

v.to_csv("/kaggle/working/validation_to_fix.csv", index=False)

66 rows need a manual label


In [3]:
from sklearn.metrics import confusion_matrix, classification_report

v = pd.read_excel("/kaggle/input/datasets/kylin0338/validationfixed/validation_fixed.xlsx")

labels = ["negative", "neutral", "positive"]
print(v["true_sentiment"].dtype)
print(v["aspect_sentiment"].dtype)

print(v["true_sentiment"].unique())
print(v["aspect_sentiment"].unique())

print(v[["true_sentiment", "aspect_sentiment"]].isna().sum())
# print(pd.DataFrame(confusion_matrix(v["true_sentiment"], v["aspect_sentiment"], labels=labels),
#                    index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels]))
# print(classification_report(v["true_sentiment"], v["aspect_sentiment"], digits=3))

object
object
['negative' 'positive' 'neutral' nan]
['negative' 'neutral' 'positive']
true_sentiment      1
aspect_sentiment    0
dtype: int64


In [4]:
v = v.dropna(subset=["true_sentiment"]).copy()

In [5]:
print(v[["true_sentiment", "aspect_sentiment"]].isna().sum())
print(v["true_sentiment"].unique())

true_sentiment      0
aspect_sentiment    0
dtype: int64
['negative' 'positive' 'neutral']


In [6]:
from sklearn.metrics import confusion_matrix, classification_report

labels = ["negative", "neutral", "positive"]

print(pd.DataFrame(
    confusion_matrix(
        v["true_sentiment"],
        v["aspect_sentiment"],
        labels=labels
    ),
    index=[f"true_{l}" for l in labels],
    columns=[f"pred_{l}" for l in labels]
))

print(classification_report(
    v["true_sentiment"],
    v["aspect_sentiment"],
    labels=labels,
    digits=3,
    zero_division=0
))

               pred_negative  pred_neutral  pred_positive
true_negative             98            18              9
true_neutral               6            27              2
true_positive              3            15            101
              precision    recall  f1-score   support

    negative      0.916     0.784     0.845       125
     neutral      0.450     0.771     0.568        35
    positive      0.902     0.849     0.874       119

    accuracy                          0.810       279
   macro avg      0.756     0.801     0.763       279
weighted avg      0.851     0.810     0.823       279



In [7]:
from transformers import pipeline

pairs = pd.read_csv("/kaggle/input/datasets/kylin0338/aspectsentimentoutput/aspect_sentiment.csv")

absa = pipeline("text-classification", model="yangheng/deberta-v3-base-absa-v1.1",
                device=0, truncation=True, max_length=256)

inputs = [{"text": s, "text_pair": a} for s, a in zip(pairs["sentence"], pairs["aspect"])]
out = absa(inputs, batch_size=64, top_k=None)

def probs(o):
    d = {x["label"].lower(): x["score"] for x in o}
    return d.get("negative", 0), d.get("neutral", 0), d.get("positive", 0)

pairs[["p_neg", "p_neu", "p_pos"]] = [probs(o) for o in out]
pairs.to_csv("/kaggle/working/pairs_with_probs.csv", index=False)

config.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/372 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

In [8]:
pairs["key"] = pairs["review_id"].astype(str) + "|" + pairs["aspect"] + "|" + pairs["sentence"]
v["key"] = v["review_id"].astype(str) + "|" + v["aspect"] + "|" + v["sentence"]
pool = pairs[~pairs["key"].isin(v["key"])].copy()

# 1) FRESH RANDOM TEST SET, label last, never inspect
test = pool.sample(150, random_state=7).assign(split="test")
pool = pool.drop(test.index)

# 2) hard-case sampling for tuning
neg_words = r"\b(?:not|n't|no|never|but|however|although|than|though|except)\b"
hard_neutral = pool[(pool["aspect_sentiment"] == "neutral") & (pool["p_neu"].between(0.4, 0.8))].sample(100, random_state=7)
pool = pool.drop(hard_neutral.index)
negation = pool[pool["sentence"].str.lower().str.contains(neg_words)].sample(100, random_state=7)
pool = pool.drop(negation.index)
weak = pool[pool["aspect"].isin(["network", "sound", "software"])].sample(100, random_state=7)

tune = pd.concat([hard_neutral.assign(split="tune_neutral"),
                  negation.assign(split="tune_negation"),
                  weak.assign(split="tune_weak")])

cols = ["key", "review_id", "sentence", "aspect", "p_neg", "p_neu", "p_pos", "aspect_sentiment", "split"]
test[cols].assign(true_sentiment="").to_csv("/kaggle/working/label_test.csv", index=False)
tune[cols].assign(true_sentiment="").to_csv("/kaggle/working/label_tune.csv", index=False)

In [9]:
tune = pd.read_excel("/kaggle/input/datasets/kylin0338/tuningandtesting/label_tune_labeled.xlsx")

# bring probabilities into the original 280 rows
v2 = v.merge(pairs[["key", "p_neg", "p_neu", "p_pos"]], on="key", how="left")
dev = pd.concat([tune, v2[["key", "sentence", "aspect", "p_neg", "p_neu", "p_pos",
                           "aspect_sentiment", "true_sentiment"]]], ignore_index=True)
dev = dev.dropna(subset=["p_neg", "true_sentiment"])
print(dev.shape, dev["true_sentiment"].value_counts())

(574, 10) true_sentiment
negative    122
Negative    118
positive    118
Positive    102
Neutral      80
neutral      34
Name: count, dtype: int64


In [10]:
dev = dev.dropna(subset=["p_neg", "true_sentiment"]).copy()

# Standardize sentiment labels
dev["true_sentiment"] = (
    dev["true_sentiment"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Verify
print(dev.shape)
print(dev["true_sentiment"].value_counts())

(574, 10)
true_sentiment
negative    240
positive    220
neutral     114
Name: count, dtype: int64


In [11]:
valid_labels = {"negative", "neutral", "positive"}

print("Unexpected labels:",
      set(dev["true_sentiment"].unique()) - valid_labels)

Unexpected labels: set()


In [12]:
from sklearn.metrics import accuracy_score, f1_score
print("baseline acc:", accuracy_score(dev["true_sentiment"], dev["aspect_sentiment"]))
print("baseline macro-F1:", f1_score(dev["true_sentiment"], dev["aspect_sentiment"], average="macro"))

baseline acc: 0.6759581881533101
baseline macro-F1: 0.6657991099779758


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold

X = dev[["p_neg", "p_neu", "p_pos"]].copy()
X = pd.concat([X, pd.get_dummies(dev["aspect"], prefix="asp")], axis=1).astype(float)
y = dev["true_sentiment"]

clf = LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced")
cv = StratifiedKFold(5, shuffle=True, random_state=0)
pred = cross_val_predict(clf, X, y, cv=cv)

print("calibrated CV acc:", accuracy_score(y, pred))
print("calibrated CV macro-F1:", f1_score(y, pred, average="macro"))

calibrated CV acc: 0.7073170731707317
calibrated CV macro-F1: 0.6843689364198365


In [14]:
def thresh_rule(r, t):
    if r["p_neu"] < t:
        return "positive" if r["p_pos"] > r["p_neg"] else "negative"
    return "neutral"

for t in [0.4, 0.5, 0.6, 0.7, 0.8]:
    p = dev.apply(lambda r: thresh_rule(r, t), axis=1)
    print(t, round(accuracy_score(y, p), 3), round(f1_score(y, p, average="macro"), 3))

0.4 0.662 0.653
0.5 0.681 0.669
0.6 0.716 0.693
0.7 0.73 0.69
0.8 0.749 0.682


In [15]:
print(dev.shape)
print(dev["split"].value_counts(dropna=False) if "split" in dev else "")
print(dev["true_sentiment"].value_counts())
print(dev["p_neg"].isna().sum())

(574, 10)
split
NaN              274
tune_neutral     100
tune_negation    100
tune_weak        100
Name: count, dtype: int64
true_sentiment
negative    240
positive    220
neutral     114
Name: count, dtype: int64
0


In [16]:
from sklearn.metrics import classification_report, f1_score, accuracy_score

def thresh_rule(r, t):
    if r["p_neu"] < t:
        return "positive" if r["p_pos"] > r["p_neg"] else "negative"
    return "neutral"

y = dev["true_sentiment"]
for t in [0.55, 0.6, 0.65, 0.7]:
    p = dev.apply(lambda r: thresh_rule(r, t), axis=1)
    print(f"\n--- t = {t} ---")
    print(classification_report(y, p, digits=3))


--- t = 0.55 ---
              precision    recall  f1-score   support

    negative      0.831     0.738     0.781       240
     neutral      0.400     0.667     0.500       114
    positive      0.854     0.664     0.747       220

    accuracy                          0.695       574
   macro avg      0.695     0.689     0.676       574
weighted avg      0.754     0.695     0.712       574


--- t = 0.6 ---
              precision    recall  f1-score   support

    negative      0.830     0.775     0.802       240
     neutral      0.435     0.649     0.521       114
    positive      0.839     0.686     0.755       220

    accuracy                          0.716       574
   macro avg      0.702     0.703     0.693       574
weighted avg      0.755     0.716     0.728       574


--- t = 0.65 ---
              precision    recall  f1-score   support

    negative      0.813     0.796     0.804       240
     neutral      0.441     0.561     0.494       114
    positive      0.81

In [17]:
import numpy as np
rng = np.random.default_rng(0)
ts = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8]
res = {t: [] for t in ts}
for _ in range(300):
    s = dev.sample(len(dev), replace=True, random_state=int(rng.integers(1e9)))
    for t in ts:
        p = s.apply(lambda r: thresh_rule(r, t), axis=1)
        res[t].append(f1_score(s["true_sentiment"], p, average="macro"))
for t in ts:
    print(t, round(np.mean(res[t]), 3), "+/-", round(np.std(res[t]), 3))

0.5 0.668 +/- 0.019
0.55 0.676 +/- 0.019
0.6 0.692 +/- 0.018
0.65 0.688 +/- 0.02
0.7 0.69 +/- 0.019
0.75 0.688 +/- 0.02
0.8 0.682 +/- 0.02


In [18]:
diffs = []
for _ in range(1000):
    s = dev.sample(len(dev), replace=True, random_state=int(rng.integers(1e9)))
    base = f1_score(s["true_sentiment"], s["aspect_sentiment"], average="macro")
    new = f1_score(s["true_sentiment"], s.apply(lambda r: thresh_rule(r, 0.65), axis=1), average="macro")
    diffs.append(new - base)
print("mean gain:", round(np.mean(diffs), 3),
      "95% interval:", np.percentile(diffs, [2.5, 97.5]).round(3))

mean gain: 0.02 95% interval: [-0.01   0.049]


In [19]:
p = dev.apply(lambda r: thresh_rule(r, 0.65), axis=1)
err = dev[(p == "neutral") & (dev["true_sentiment"] != "neutral")]
print(err["aspect"].value_counts())
for _, r in err.sample(15, random_state=2).iterrows():
    print(f"[true={r['true_sentiment']}] {r['aspect']}: {r['sentence'][:140]}")

aspect
software            25
network             12
battery             11
screen               7
camera               7
storage              5
price                4
sound                3
customer_service     3
delivery             2
build                2
Name: count, dtype: int64
[true=negative] screen: Cons: - When you double press the home screen, it pops up "My Magazine" app by default.
[true=positive] battery: Not factory charger, much better charge cable.
[true=negative] customer_service: Is that a US version or Asian version oneplus 6t and why no US warranty
[true=negative] sound: The screen vibrates with an annoying rattle if you play audio.
[true=negative] software: This phone is yet to get an upgrade to Android O. Also, surprisingly this phone does not come with an ordinary earphone in the original box.
[true=positive] software: I used the app mobiletrans which seemed to work pretty well with this phone.
[true=positive] software: You touch an app and it opens.
[true=posit

In [20]:
import pandas as pd, numpy as np
from sklearn.metrics import classification_report, f1_score, accuracy_score

def make_key(d):
    return d["review_id"].astype(str) + "|" + d["aspect"] + "|" + d["sentence"]

pairs = pd.read_csv("/kaggle/input/datasets/kylin0338/pairsversion2/pairs_with_probs_v2.csv")
pairs["key"] = make_key(pairs)

v = pd.read_excel("/kaggle/input/datasets/kylin0338/validationfixed/validation_fixed.xlsx")
v["key"] = make_key(v)
tune = pd.read_excel("/kaggle/input/datasets/kylin0338/tuningandtesting/label_tune_labeled.xlsx")

cols = ["key", "aspect", "sentence", "true_sentiment"]
labeled = pd.concat([v[cols].assign(src="original"), tune[cols].assign(src="tune")], ignore_index=True)
labeled["true_sentiment"] = labeled["true_sentiment"].replace("", np.nan)
labeled = labeled.dropna(subset=["true_sentiment"]).drop_duplicates("key")
print("labeled rows before the matcher fix:", len(labeled))

# an inner join keeps only labeled pairs that still exist after the matcher fix
dev2 = labeled.merge(
    pairs[["key", "p_neg", "p_neu", "p_pos"]].drop_duplicates("key"), on="key", how="inner")
dev2["aspect_sentiment"] = np.array(["negative", "neutral", "positive"])[
    dev2[["p_neg", "p_neu", "p_pos"]].values.argmax(axis=1)]

print(f"kept {len(dev2)} of {len(labeled)} labeled rows ({len(labeled) - len(dev2)} dropped by the matcher fix)")
print(dev2["true_sentiment"].value_counts())

labeled rows before the matcher fix: 579
kept 501 of 579 labeled rows (78 dropped by the matcher fix)
true_sentiment
negative    116
positive    112
Negative     91
Positive     83
Neutral      67
neutral      32
Name: count, dtype: int64


In [21]:
labeled["true_sentiment"] = labeled["true_sentiment"].astype(str).str.strip().str.lower()
print(labeled["true_sentiment"].value_counts())   # should show exactly 3 classes

true_sentiment
negative    243
positive    221
neutral     115
Name: count, dtype: int64


In [22]:
dev2["true_sentiment"] = dev2["true_sentiment"].astype(str).str.strip().str.lower()
print(dev2["true_sentiment"].value_counts())      # must show exactly 3 classes
print(dev2["true_sentiment"].unique())            # check for typos like "positve" or "mixed"

true_sentiment
negative    207
positive    195
neutral      99
Name: count, dtype: int64
['negative' 'positive' 'neutral']


In [23]:
y = dev2["true_sentiment"]
print("baseline  acc", round(accuracy_score(y, dev2["aspect_sentiment"]), 3),
      "macro-F1", round(f1_score(y, dev2["aspect_sentiment"], average="macro"), 3))
for t in [0.55, 0.6, 0.65, 0.7, 0.75]:
    p = dev2.apply(lambda r: thresh_rule(r, t), axis=1)
    print(f"T={t}  acc", round(accuracy_score(y, p), 3), "macro-F1", round(f1_score(y, p, average="macro"), 3))
print(classification_report(y, dev2["aspect_sentiment"], digits=3, zero_division=0))

baseline  acc 0.693 macro-F1 0.679
T=0.55  acc 0.715 macro-F1 0.691
T=0.6  acc 0.735 macro-F1 0.707
T=0.65  acc 0.739 macro-F1 0.701
T=0.7  acc 0.745 macro-F1 0.699
T=0.75  acc 0.747 macro-F1 0.686
              precision    recall  f1-score   support

    negative      0.860     0.739     0.795       207
     neutral      0.403     0.758     0.526        99
    positive      0.869     0.610     0.717       195

    accuracy                          0.693       501
   macro avg      0.710     0.702     0.679       501
weighted avg      0.773     0.693     0.711       501



In [24]:
labs = np.array(["negative", "neutral", "positive"])
rule = np.where(dev2["p_neu"] < 0.65,
                np.where(dev2["p_pos"] > dev2["p_neg"], "positive", "negative"),
                "neutral")
y = dev2["true_sentiment"].values
base = dev2["aspect_sentiment"].values

rng = np.random.default_rng(0)
diffs = []
for _ in range(1000):
    i = rng.integers(0, len(y), len(y))
    diffs.append(f1_score(y[i], rule[i], average="macro") - f1_score(y[i], base[i], average="macro"))
print("mean gain:", round(np.mean(diffs), 3), "95% interval:", np.percentile(diffs, [2.5, 97.5]).round(3))

mean gain: 0.021 95% interval: [-0.011  0.054]


In [25]:
seen = set(labeled["key"]) | set(tune["key"])
pool = pairs[~pairs["key"].isin(seen)].copy()
test = pool.sample(200, random_state=11).reset_index(drop=True)

# file you label: no model columns
label = test[["key", "review_id", "aspect", "sentence"]].copy()
label["aspect_relevant"] = ""    # 1 if the sentence really expresses something about this aspect, else 0
label["true_sentiment"] = ""     # lowercase: positive / negative / neutral, only if aspect_relevant = 1
label.to_csv("/kaggle/working/label_test_v2.csv", index=False)

# kept aside, merged back only after you finish labeling
test[["key", "p_neg", "p_neu", "p_pos", "aspect_sentiment"]].to_csv(
    "/kaggle/working/test_predictions_hidden.csv", index=False)
print(test["aspect"].value_counts())

aspect
battery             37
network             25
screen              20
camera              19
price               16
performance         12
design              12
delivery            12
customer_service    12
software            10
storage              8
usability            7
build                6
sound                4
Name: count, dtype: int64


In [26]:
import pandas as pd, numpy as np
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report)

lab = pd.read_csv("/kaggle/input/datasets/kylin0338/labeltestv2/label_test_v2_LABELED.csv")
hid = pd.read_csv("/kaggle/working/test_predictions_hidden.csv")

# normalize labels
lab["aspect_relevant"] = pd.to_numeric(lab["aspect_relevant"], errors="coerce")
lab["true_sentiment"] = lab["true_sentiment"].astype(str).str.strip().str.lower()
lab.loc[lab["true_sentiment"].isin(["nan", ""]), "true_sentiment"] = np.nan

# checks
print("rows:", len(lab), "| missing aspect_relevant:", lab["aspect_relevant"].isna().sum())
print("relevant=1 but no sentiment:", ((lab["aspect_relevant"] == 1) & lab["true_sentiment"].isna()).sum())
print("sentiment labels:", lab["true_sentiment"].value_counts().to_dict())

bad = lab[lab["true_sentiment"].notna() & ~lab["true_sentiment"].isin(["positive", "negative", "neutral"])]
print("unexpected label values:", bad["true_sentiment"].unique())

rows: 200 | missing aspect_relevant: 0
relevant=1 but no sentiment: 0
sentiment labels: {'positive': 82, 'negative': 66, 'neutral': 9}
unexpected label values: []


In [27]:
d = lab.merge(hid, on="key", how="inner")
print("merged rows:", len(d))        # should equal len(lab)

# aspect precision on a random sample
n, k = len(d), int(d["aspect_relevant"].sum())
p = k / n
se = np.sqrt(p * (1 - p) / n)
print(f"\nAspect precision: {p:.3f}  (95% interval about {p - 1.96*se:.3f} to {p + 1.96*se:.3f}, n={n})")
print(d.groupby("aspect")["aspect_relevant"].agg(["mean", "count"]).round(2).sort_values("mean"))

merged rows: 200

Aspect precision: 0.785  (95% interval about 0.728 to 0.842, n=200)
                  mean  count
aspect                       
customer_service  0.50     12
battery           0.62     37
design            0.67     12
network           0.72     25
camera            0.79     19
build             0.83      6
usability         0.86      7
price             0.88     16
storage           0.88      8
delivery          0.92     12
performance       0.92     12
screen            0.95     20
sound             1.00      4
software          1.00     10


In [28]:
r = d[(d["aspect_relevant"] == 1) & d["true_sentiment"].notna()].copy()
print("relevant rows used for sentiment:", len(r))
print(r["true_sentiment"].value_counts().to_dict())

T = 0.65   # fixed in advance
r["rule_pred"] = np.where(r["p_neu"] < T,
                          np.where(r["p_pos"] > r["p_neg"], "positive", "negative"),
                          "neutral")
y = r["true_sentiment"].values
base = r["aspect_sentiment"].values
rule = r["rule_pred"].values
labels = ["negative", "neutral", "positive"]

for name, pred in [("baseline", base), (f"threshold T={T}", rule)]:
    print(f"\n=== {name} ===")
    print("accuracy:", round(accuracy_score(y, pred), 3),
          "| macro-F1:", round(f1_score(y, pred, average="macro"), 3))
    print(pd.DataFrame(confusion_matrix(y, pred, labels=labels),
                       index=[f"true_{l}" for l in labels],
                       columns=[f"pred_{l}" for l in labels]))
    print(classification_report(y, pred, labels=labels, digits=3, zero_division=0))

relevant rows used for sentiment: 157
{'positive': 82, 'negative': 66, 'neutral': 9}

=== baseline ===
accuracy: 0.828 | macro-F1: 0.667
               pred_negative  pred_neutral  pred_positive
true_negative             57             5              4
true_neutral               4             3              2
true_positive              5             7             70
              precision    recall  f1-score   support

    negative      0.864     0.864     0.864        66
     neutral      0.200     0.333     0.250         9
    positive      0.921     0.854     0.886        82

    accuracy                          0.828       157
   macro avg      0.662     0.684     0.667       157
weighted avg      0.856     0.828     0.840       157


=== threshold T=0.65 ===
accuracy: 0.828 | macro-F1: 0.645
               pred_negative  pred_neutral  pred_positive
true_negative             58             4              4
true_neutral               4             2              3
true_positive   

In [29]:
rng = np.random.default_rng(0)
diffs = []
for _ in range(2000):
    i = rng.integers(0, len(y), len(y))
    diffs.append(f1_score(y[i], rule[i], average="macro") - f1_score(y[i], base[i], average="macro"))
lo, hi = np.percentile(diffs, [2.5, 97.5])

f1_base = f1_score(y, base, average="macro")
f1_rule = f1_score(y, rule, average="macro")
print(f"macro-F1 baseline {f1_base:.3f} | rule {f1_rule:.3f} | gain {f1_rule - f1_base:+.3f}")
print(f"95% bootstrap interval for the gain: [{lo:.3f}, {hi:.3f}]")

USE_RULE = f1_rule >= f1_base      # the rule written down before testing
print("\nDecision:", "use the threshold rule" if USE_RULE else "keep the baseline")
if USE_RULE and lo <= 0:
    print("(the gain is not statistically distinguishable from zero; describe it that way)")

macro-F1 baseline 0.667 | rule 0.645 | gain -0.022
95% bootstrap interval for the gain: [-0.080, 0.015]

Decision: keep the baseline


In [30]:
import pandas as pd

pairs = pd.read_csv("/kaggle/input/datasets/kylin0338/pairsversion2/pairs_with_probs_v2.csv")

# the decision from the blind test: keep the baseline (the model's top label)
pairs["sentiment_final"] = pairs["aspect_sentiment"]

print(pairs["sentiment_final"].value_counts(normalize=True).round(3))

keep = ["review_id", "asin", "rating", "sentence", "aspect", "p_neg", "p_neu", "p_pos",
        "aspect_sentiment", "sentiment_final"]
pairs[keep].to_csv("/kaggle/working/aspect_sentiment_final.csv", index=False)
print("saved", len(pairs), "rows")

sentiment_final
positive    0.406
negative    0.371
neutral     0.223
Name: proportion, dtype: float64
saved 43230 rows
